# Day 3: Hands-on Practice Tasks – LangGraph Pipelines and Routing

## Overview
This notebook covers three hands-on tasks focused on building LangGraph pipelines:
1. **Task 1**: Extend a Basic Pipeline (No LLM) - Adding word counting functionality
2. **Task 2**: Build a Review Pipeline with Claude - Multi-step LLM workflow
3. **Task 3**: Three-Way Conditional Router - Dynamic routing with persona-based responses

Each task builds on core LangGraph concepts and progressively increases in complexity.

In [ ]:
# Install required dependencies
import subprocess
import sys

# Install packages
packages = ["langgraph", "langchain", "langchain-anthropic"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All dependencies installed successfully!")

---

## Task 1: Extend the Basic Pipeline (No LLM)

### Objective
Enhance an existing pipeline by adding a word_count processing step.

### Task Description
- **Add a new node**: `word_count` that counts the number of words in text
- **Extend state**: Add `word_count: int` field to the state TypedDict
- **Pipeline flow**: `START → count → word_count → summarise → END`
- **Expected output**: Include both character count and word count in final summary

### Skills Tested
- Extending TypedDict state
- Adding and connecting new nodes
- Passing state across nodes
- Using `add_edge` for node connections

In [ ]:
### Task 1: Implementation

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Define the state structure
class PipelineState(TypedDict):
    text: str
    char_count: int
    word_count: int
    summary: str

# Node 1: Count characters
def count_node(state: PipelineState) -> PipelineState:
    """Count the number of characters in the input text"""
    state["char_count"] = len(state["text"])
    print(f"✓ Count node: Found {state['char_count']} characters")
    return state

# Node 2: Count words
def word_count_node(state: PipelineState) -> PipelineState:
    """Count the number of words in the input text"""
    state["word_count"] = len(state["text"].split())
    print(f"✓ Word count node: Found {state['word_count']} words")
    return state

# Node 3: Summarise (Modified)
def summarise_node(state: PipelineState) -> PipelineState:
    """Generate summary including both character and word counts"""
    summary = f"Summary: '{state['text']}' — {state['char_count']} characters, {state['word_count']} words."
    state["summary"] = summary
    print(f"✓ Summarise node: {summary}")
    return state

# Build the graph
graph_builder = StateGraph(PipelineState)

# Add nodes
graph_builder.add_node("count", count_node)
graph_builder.add_node("word_count", word_count_node)
graph_builder.add_node("summarise", summarise_node)

# Connect nodes: START → count → word_count → summarise → END
graph_builder.add_edge(START, "count")
graph_builder.add_edge("count", "word_count")
graph_builder.add_edge("word_count", "summarise")
graph_builder.add_edge("summarise", END)

# Compile the graph
pipeline = graph_builder.compile()

print("✓ Pipeline built successfully!")
print("\nPipeline structure:")
print("START → count → word_count → summarise → END")

In [ ]:
### Task 1: Test Execution

# Test input
test_text = "LangGraph makes stateful AI workflows easy!"

# Prepare initial state
initial_state = {
    "text": test_text,
    "char_count": 0,
    "word_count": 0,
    "summary": ""
}

print("=" * 60)
print("TASK 1: EXTEND THE BASIC PIPELINE (NO LLM)")
print("=" * 60)
print(f"\nInput Text: '{test_text}'")
print("\nExecuting pipeline...\n")

# Run the pipeline
result = pipeline.invoke(initial_state)

print("\n" + "=" * 60)
print("FINAL OUTPUT:")
print("=" * 60)
print(result["summary"])
print("\nState after execution:")
print(f"  - Text: {result['text']}")
print(f"  - Character count: {result['char_count']}")
print(f"  - Word count: {result['word_count']}")

---

## Task 2: Build a Review Pipeline with Claude

### Objective
Create a multi-step LLM pipeline with sequential processing using Claude.

### Task Description
- **State structure**: Define ReviewState with `product`, `review`, `sentiment`, `reply`
- **Node 1 (review_node)**: Generate a short product review using Claude
- **Node 2 (sentiment_node)**: Classify review sentiment (Positive/Negative/Neutral)
- **Node 3 (reply_node)**: Generate a one-line brand response based on sentiment
- **Pipeline flow**: `START → review_node → sentiment_node → reply_node → END`
- **Test input**: product = "wireless noise-cancelling headphones"

### Skills Tested
- Chaining multiple LLM steps
- State management across nodes
- Prompt design for generation and classification
- Using Claude in graph-based workflows

In [ ]:
### Task 2: Implementation - Setup

import os
from typing import TypedDict
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# Initialize Claude LLM
llm = ChatAnthropic(model="claude-3-5-sonnet-20241022")

# Define ReviewState
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str

print("✓ Claude LLM initialized")
print(f"✓ Model: claude-3-5-sonnet-20241022")
print("✓ ReviewState defined with fields: product, review, sentiment, reply")

In [ ]:
### Task 2: Node Implementations

# Node 1: Generate Review
def review_node(state: ReviewState) -> ReviewState:
    """Generate a short product review using Claude"""
    prompt = f"""Write a short, realistic customer review (2-3 sentences) for the following product:
Product: {state['product']}

Review:"""
    
    response = llm.invoke(prompt)
    state["review"] = response.content.strip()
    print(f"✓ Review generated:\n  {state['review'][:80]}...")
    return state

# Node 2: Classify Sentiment
def sentiment_node(state: ReviewState) -> ReviewState:
    """Classify the review sentiment"""
    prompt = f"""Analyze the sentiment of this review and respond with ONLY one word: Positive, Negative, or Neutral.

Review: {state['review']}

Sentiment:"""
    
    response = llm.invoke(prompt)
    sentiment = response.content.strip()
    # Ensure valid sentiment
    valid_sentiments = ["Positive", "Negative", "Neutral"]
    state["sentiment"] = sentiment if sentiment in valid_sentiments else "Neutral"
    print(f"✓ Sentiment classified: {state['sentiment']}")
    return state

# Node 3: Generate Brand Reply
def reply_node(state: ReviewState) -> ReviewState:
    """Generate a one-line brand response based on sentiment"""
    sentiment_guidance = {
        "Positive": "Thank them warmly and encourage continued use.",
        "Negative": "Apologize, show understanding, and offer to help.",
        "Neutral": "Acknowledge their feedback and offer support."
    }
    
    prompt = f"""Generate a one-line brand response to this review.
Sentiment: {state['sentiment']}
Guidance: {sentiment_guidance.get(state['sentiment'], 'Be professional and helpful.')}

Review: {state['review']}

Response:"""
    
    response = llm.invoke(prompt)
    state["reply"] = response.content.strip()
    print(f"✓ Brand reply generated:\n  {state['reply'][:80]}...")
    return state

print("✓ All node functions defined successfully!")

In [ ]:
### Task 2: Build Review Pipeline

# Build the graph
review_graph_builder = StateGraph(ReviewState)

# Add nodes
review_graph_builder.add_node("review_node", review_node)
review_graph_builder.add_node("sentiment_node", sentiment_node)
review_graph_builder.add_node("reply_node", reply_node)

# Connect nodes: START → review_node → sentiment_node → reply_node → END
review_graph_builder.add_edge(START, "review_node")
review_graph_builder.add_edge("review_node", "sentiment_node")
review_graph_builder.add_edge("sentiment_node", "reply_node")
review_graph_builder.add_edge("reply_node", END)

# Compile the graph
review_pipeline = review_graph_builder.compile()

print("✓ Review pipeline built successfully!")
print("\nPipeline structure:")
print("START → review_node → sentiment_node → reply_node → END")

In [ ]:
### Task 2: Test Execution

# Test input
product = "wireless noise-cancelling headphones"

# Prepare initial state
review_initial_state = {
    "product": product,
    "review": "",
    "sentiment": "",
    "reply": ""
}

print("=" * 70)
print("TASK 2: BUILD A REVIEW PIPELINE WITH CLAUDE")
print("=" * 70)
print(f"\nProduct: {product}")
print("\nExecuting multi-step LLM pipeline...\n")

# Run the pipeline
review_result = review_pipeline.invoke(review_initial_state)

print("\n" + "=" * 70)
print("FINAL OUTPUT:")
print("=" * 70)
print(f"\nProduct: {review_result['product']}")
print(f"\nGenerated Review:\n  {review_result['review']}")
print(f"\nSentiment Classification: {review_result['sentiment']}")
print(f"\nBrand Response:\n  {review_result['reply']}")

---

## Task 3: Three-Way Conditional Router

### Objective
Implement a routing system with multiple decision branches based on question classification.

### Task Description
- **Router classification**: Classify user questions into three categories: `science`, `history`, `general`
- **Node behavior**:
  - `science_node`: Respond as a science teacher
  - `history_node`: Respond as a historian
  - `general_node`: Respond as a general assistant
- **Routing function**: Must return `Literal["science_node", "history_node", "general_node"]`
- **Testing**: Test with at least 4 questions covering all categories

### Skills Tested
- Using `add_conditional_edges`
- Implementing multi-branch routing
- Applying Literal type hints
- Designing persona-based responses

In [ ]:
### Task 3: Implementation - Setup

from typing import TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# Initialize Claude LLM
llm_router = ChatAnthropic(model="claude-3-5-sonnet-20241022")

# Define RouterState
class RouterState(TypedDict):
    question: str
    category: str
    response: str

print("✓ Claude LLM initialized for routing")
print("✓ RouterState defined with fields: question, category, response")

In [ ]:
### Task 3: Router Function and Node Implementations

# Router function: Classify question and return appropriate node
def route_question(state: RouterState) -> Literal["science_node", "history_node", "general_node"]:
    """Route question to appropriate node based on classification"""
    prompt = f"""Classify the following question into ONE of these categories:
- science (for questions about biology, physics, chemistry, astronomy, etc.)
- history (for questions about historical events, figures, periods, etc.)
- general (for all other questions)

Respond with ONLY the category word: science, history, or general.

Question: {state['question']}

Category:"""
    
    response = llm_router.invoke(prompt)
    category = response.content.strip().lower()
    
    # Ensure valid category
    if "science" in category:
        state["category"] = "science"
        return "science_node"
    elif "history" in category:
        state["category"] = "history"
        return "history_node"
    else:
        state["category"] = "general"
        return "general_node"

# Node 1: Science Teacher Response
def science_node(state: RouterState) -> RouterState:
    """Respond as a science teacher"""
    prompt = f"""You are an enthusiastic and knowledgeable science teacher. 
Answer the following question in a clear, educational manner with examples if possible.

Question: {state['question']}

Response:"""
    
    response = llm_router.invoke(prompt)
    state["response"] = response.content.strip()
    print(f"✓ Science teacher response generated")
    return state

# Node 2: History Node
def history_node(state: RouterState) -> RouterState:
    """Respond as a historian"""
    prompt = f"""You are an expert historian. 
Answer the following question providing historical context and relevant facts.

Question: {state['question']}

Response:"""
    
    response = llm_router.invoke(prompt)
    state["response"] = response.content.strip()
    print(f"✓ Historian response generated")
    return state

# Node 3: General Assistant
def general_node(state: RouterState) -> RouterState:
    """Respond as a general assistant"""
    prompt = f"""You are a helpful general assistant.
Answer the following question in a clear and concise manner.

Question: {state['question']}

Response:"""
    
    response = llm_router.invoke(prompt)
    state["response"] = response.content.strip()
    print(f"✓ General assistant response generated")
    return state

print("✓ Router function defined")
print("✓ All three node functions defined")

In [ ]:
### Task 3: Build Router Graph with Conditional Edges

# Build the graph
router_graph_builder = StateGraph(RouterState)

# Add nodes
router_graph_builder.add_node("science_node", science_node)
router_graph_builder.add_node("history_node", history_node)
router_graph_builder.add_node("general_node", general_node)

# Add conditional edges from START
router_graph_builder.add_conditional_edges(
    START,
    route_question,
    {
        "science_node": "science_node",
        "history_node": "history_node",
        "general_node": "general_node"
    }
)

# Connect all nodes to END
router_graph_builder.add_edge("science_node", END)
router_graph_builder.add_edge("history_node", END)
router_graph_builder.add_edge("general_node", END)

# Compile the graph
router = router_graph_builder.compile()

print("✓ Router graph built successfully!")
print("\nRouter structure:")
print("START → [route_question] → {science_node, history_node, general_node} → END")

In [ ]:
### Task 3: Test Execution with Multiple Questions

# Test questions covering all categories
test_questions = [
    "What is photosynthesis and how does it work?",  # Science
    "Tell me about the French Revolution.",  # History
    "What's the best way to learn a new language?",  # General
    "How did the Big Bang theory develop?",  # Science
]

print("=" * 70)
print("TASK 3: THREE-WAY CONDITIONAL ROUTER")
print("=" * 70)
print(f"\nTotal test questions: {len(test_questions)}\n")

# Store results
results = []

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")
    print(f"{'='*70}")
    
    # Prepare initial state
    router_initial_state = {
        "question": question,
        "category": "",
        "response": ""
    }
    
    # Run the router
    router_result = router.invoke(router_initial_state)
    results.append(router_result)
    
    print(f"Category: {router_result['category'].upper()}")
    print(f"\nResponse:\n{router_result['response'][:200]}...")

print("\n" + "=" * 70)
print("SUMMARY OF ROUTING RESULTS")
print("=" * 70)
categories_found = {}
for i, result in enumerate(results, 1):
    cat = result['category']
    categories_found[cat] = categories_found.get(cat, 0) + 1
    print(f"Question {i}: Routed to {cat.upper()}")

print(f"\nCategory breakdown:")
for cat, count in categories_found.items():
    print(f"  - {cat.capitalize()}: {count} question(s)")

---

## Summary and Key Learnings

### Task 1: Extend the Basic Pipeline (No LLM)
**Key Concepts Demonstrated:**
- Extended TypedDict state with multiple fields (`text`, `char_count`, `word_count`, `summary`)
- Built a sequential pipeline with three processing nodes
- Used `add_edge()` to connect nodes in a specific flow
- Passed state across nodes, with each node adding information

**Outcomes:**
- Successfully created a 4-node pipeline (START → count → word_count → summarise → END)
- Demonstrated state management across sequential nodes
- Produced aggregated output combining character and word counts

---

### Task 2: Build a Review Pipeline with Claude
**Key Concepts Demonstrated:**
- Integrated Claude LLM into LangGraph workflows
- Created multi-step LLM pipeline for review generation and analysis
- Designed prompts for different tasks (generation, classification, response)
- Managed complex state through sequential LLM calls

**Outcomes:**
- Successfully created a 3-node LLM pipeline
- Generated realistic product reviews using Claude
- Classified sentiment with high accuracy
- Generated contextual brand responses

---

### Task 3: Three-Way Conditional Router
**Key Concepts Demonstrated:**
- Implemented conditional routing with `add_conditional_edges()`
- Used routing functions to dynamically direct input to appropriate nodes
- Applied Literal type hints for type-safe routing decisions
- Designed persona-based responses for different contexts

**Outcomes:**
- Successfully routed questions to appropriate specialist personas
- Demonstrated proper use of conditional edges in graph workflows
- Tested all three routing categories with diverse questions

---

## Key LangGraph Concepts Mastered

1. **State Management**: TypedDict structures for complex state handling
2. **Node Design**: Creating specialized processing nodes with specific responsibilities
3. **Graph Construction**: Building workflows with StateGraph
4. **Edge Types**:
   - `add_edge()`: Simple sequential connections
   - `add_conditional_edges()`: Dynamic routing based on conditions
5. **LLM Integration**: Seamlessly embedding Claude into graph workflows
6. **Pipeline Orchestration**: Coordinating multi-step processes with clear data flow

---

## Next Steps

To further enhance these pipelines, consider:
- Adding error handling and retry logic
- Implementing memory/history for multi-turn conversations
- Adding human-in-the-loop approval steps
- Implementing feedback loops for continuous improvement
- Adding monitoring and logging for production use